<a href="https://colab.research.google.com/github/columbia-data-club/meetings/blob/main/2025/april_16_numpy_xarray.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![A blue background with the pandas logo and the words Columbia Data Club on it](https://raw.githubusercontent.com/columbia-data-club/meetings/main/assets/images/2025/xarray.png)

# NumPy and Xarray

April 16, 2025

by [Moacir P. de Sá Pereira](https://moacir.com) for the [Columbia Data Club](https://github.com/columbia-data-club/)

A basic understanding of Python syntax (such as the one covered in the Data Club’s [Intro to Python video](https://youtu.be/l45rzo4MUHs)) should suffice for this notebook.


## Why?

Today’s notebook will move more slowly than previous ones this semester and will not begin with introducing a bunch of data, mostly because in thinking about why I wanted to address this topic, a large part of my interest was in clarifying for myself why on earth I care about [NumPy](https://numpy.org/) and [Xarray](https://docs.xarray.dev/) in the first place.

After all, we already have libraries like pandas and Polars to help us with our data wrangling? And both already use NumPy at least to some degree on the inside, so why worry about NumPy per se? And what, after all, even is Xarray?

I think I have a few straightforward answers about NumPy. Xarray will be a bit trickier.

1. NumPy is the backbone for numerical computation in Python. In short, even if you use libraries like pandas and Polars, you are still likely to have to think something through in NumPy terms. Despite having its own `pd.NA` object, pandas data frames often feature `np.nan`. Polars [treats `null` and `NaN` differently](https://docs.pola.rs/user-guide/expressions/missing-data/#not-a-number-or-nan-values), but the latter is still in the mix. This is a silly example that was the first one I thought of, but using a data-wrangling library like pandas or Polars without at least basic knowledge of NumPy is very limiting (I know this first hand!)

2. NumPy syntax and functions are inescapable in machine learning applications. The multi-dimensional and vectorized nature of machine learning means that a library that can handle both multi-dimensionality and vectorization is central to libraries like sci-kit learn, pyTorch, and TensorFlow. Furthermore, at least in my experience, a lot of the debugging around ML applications comes down to handling NumPy arrays.

3. Fast.

## OK, but Why Xarray?

I'll show rather than tell the benefits of using Xarray below, but it’s helpful to understand that Xarray is still, itself, built atop NumPy. The immediate wins Xarray provides is adding labeling to NumPy’s $n$-dimensional arrays (tensors), helping us align our data and view our data through sensible labels. A lot of this can be done of course by zipping a `labels` list alongside some vector of values, but this stabilizes that kind of behavior. Then it turns it into a powerful interface for interacting with the data. Real-world data is not pure vectors, matrices, and tensors; the labeling is an important means by which we organize the data and its gaps, and that is what Xarray provides.

Furthermore, not only are the data labeled, but so, too, are the dimensions, meaning `axis=0` is a thing of the past.

Xarray especially shines in more-than-two dimensional datasets that can start to look somewhat like a relational database, as we'll see below.

Nevertheless, the classic example is a [NetCDF](https://en.wikipedia.org/wiki/NetCDF) file full of climate data. Every vector of instrument readings (temperature, pressure, and wind-speed, say) is located in three dimensions: latitude, longitude, and time. Try to imagine working on this sort of thing in pandas without creating a very, very long dataframe.

## NumPy Cheatsheet

### Lists and Arrays


First, there is the conceptual issue of the core NumPy data structure, the numpy `array`. It differs from a Python list in two vital ways: the length of an array is fixed at the moment of instantiation and its values all have to be the same data type

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import holoviews as hv
from holoviews import opts
hv.extension('bokeh')
import hvplot.xarray
import xarray as xr
import polars as pl

l = [1, 2, 3, "four"]
a = np.array([1, 2, 3, "four"])

In [ ]:
l[0] + l[1]

In [ ]:
a[0] + a[1]

In [ ]:
a

In [ ]:
l.append(5)
l

In [ ]:
np.append(a, 5)
a

In [ ]:
aa = np.append(a, 5)
aa

But you can use `np.insert` and `np.delete` to actually change arrays. To me this feels a little like an anti-pattern. List methods like `append` and `pop` are $O(1)$, so lists make a lot more sense for adding and deleting values. In contrast, adding a value to the end of a NumPy array is $O(n)$ because it recreates the array every time.

### Slicing and Dicing

NumPy relies on Python’s flexible subscripting to let us chop up arrays. This comes in handy particularly when reshaping tensors. In the subscript `[i, j, k]`, `i`, `j`, and `k` refer to the 0th, 1th, and 2th axes (or dimensions). If the array is only two dimensions, then including `k` will throw.

`:` is "all."

In [ ]:
a = np.array([[1, 2, 3, 4, 5, 6], [7, 8, 9, 10, 11, 12]])
a

In [ ]:
a[1,1]

Confusingly, `:` also introduces the slice syntax, `[i:j:k]`, where `i` is the starting index, `j` is the (unincluded) ending index, and `k` is the step size.

In [ ]:
a[:,0:4:2]

In [ ]:
a[1,1:4:2]

### Initializing Arrays

You can initialize arrays with preset values or with lists, as above.

In [ ]:
empty = np.empty([2,3])
linear = np.linspace(0, 10, 5)
arange = np.arange(0, 10, 2)
random = np.random.random([2,3])
zeros = np.zeros([2,2])
eye = np.eye(4)

print("Empty:")
print(empty)

print("\nLinear space (start, end, number of splits):")
print(linear)

print("\nRange (start, end, step size):")
print(arange)

print("\nRandom:")
print(random)

print("\nZeros:")
print(zeros)

print("\nIdentity:")
print(eye)

## Xarray

Let’s start with a toy dataset. Suppose I have four restaurants, one in each important borough, and the restaurants have four items on the menu.

In [ ]:
restaurants = ["Manhattan", "Bronx", "Queens", "Brooklyn"]
items = ["burrito", "taco", "sope", "huarache"]

The restaurants were open on every business day in January except New Years and MLK Day.

In [ ]:
full_dates = (
    pl.date_range(
        start=pl.date(2025, 1, 1),
        end=pl.date(2025, 1, 31),
        interval="1d",
        eager=True,
    )
)

holidays = pl.Series(["2025-01-01", "2025-01-20"]).str.to_date()

dates = full_dates.filter(full_dates.dt.weekday() <= 5)
dates = dates.filter(~dates.is_in(holidays))
dates.head(5)

We sell items in a Poisson distribution, so let’s generate a random $4 \times 4$ matrix for our lambdas.

In [ ]:
lambdas = np.random.randint(10, 30, size=(len(restaurants), len(items)))
lambdas

But let’s goose the numbers so that the Manhattan branch is the most popular, in general.

In [ ]:
store_multipliers = np.array([3, 2, 1, 0.8])
lambdas * store_multipliers

Wait.

In [ ]:
lambdas * store_multipliers.T

This issue pertains to [NumPy broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) In general, NumPy tries to guess correctly how to transform arrays that are different dimensions, but some times it does not work quite right, especially since a one dimensional array like `store_multipliers` is not actually a $4\times 1$ matrix in the way we treat vectors like matrices when working with them.

In [ ]:
print(store_multipliers.shape)
print(store_multipliers.T.shape)
store_multipliers

In [ ]:
store_multipliers_column_vector = store_multipliers[:, None]
print(store_multipliers_column_vector.shape)
store_multipliers_column_vector

In [ ]:
(lambdas * store_multipliers_column_vector).astype(int)

Each row is a restaurant, and each column is an item on the menu. Now let’s use these lambdas to generate sales data.

In [ ]:
sales_data = np.random.poisson(lambdas)
sales_data

OK, but we want sales data for all of our days. Add a dimension to `lambdas`, but notice that we're using the ellipsis operator `...` to stand in for "all of the (remaining) dimensions.

In [ ]:
print(lambdas.shape)
print(lambdas[..., None].shape)
lambdas[..., None]

Now even though this third dimension has a length of only one, NumPy will let us strech it to the length of the number of days we have.

In [ ]:
sales_data = np.random.poisson(lambdas[..., None], size=(len(restaurants), len(items), len(dates)))
print(sales_data.shape)
sales_data

Let’s derive one more list of lambdas to measure daily walk-ins. These people may just get a coffee, which we do not track.

In [ ]:
walk_ins = np.random.randint(30, 60, size=len(restaurants))
print(walk_ins)
walk_ins = (walk_ins * store_multipliers).astype(int)
print(walk_ins)

In [ ]:
walk_in_data = np.random.poisson(walk_ins[:, None], size=(len(restaurants), len(dates)))
print(walk_in_data.shape)
walk_in_data

Now that we’ve had this opportunity to do all this silly extra stuff that nevertheless showed us a lot about how NumPy works, let’s make our xarray data set.

In [ ]:
ds = xr.Dataset(
    {
        'sales': (['restaurant', 'item', 'date'], sales_data),
        'walk_ins': (['restaurant', 'date'], walk_in_data)
    },
    coords={
        'restaurant': restaurants,
        'item': items,
        'date': dates,
        'calendar_date': full_dates
    },
    attrs={
        'description': 'Simulated daily sales with Poisson distribution',
        'time_period': 'January 2025'
    }
)

ds

### Inspecting our Dataset

Now lets have fun!

In [ ]:
ds.sales.sel(restaurant='Manhattan').plot.scatter(x='date', col='item', col_wrap=2)


In [ ]:
ds.sales.sel(restaurant='Manhattan').hvplot.bar(
    x='date',
    y='sales',
    col='item',
    width=2000,
    height=500,
    rot=45,
)

In [ ]:
ds.sales.mean('date').plot.imshow(x='item', y='restaurant')


In [ ]:
# Add day of week coordinate
ds.coords['dayofweek'] = ds.date.dt.dayofweek

# Plot
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=ds.sales.to_dataframe(),
    x='dayofweek',
    y='sales',
    hue='restaurant',
    palette='Set2'
)
plt.title('Sales Distribution by Weekday')
plt.xticks(range(7), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
plt.legend(bbox_to_anchor=(1.05, 1))

In [ ]:
ds.sales.to_dataframe()

In [ ]:
corr_matrix = xr.Dataset({
    'item_sales': ds.sales.sum('restaurant'),
    'walkins': ds.walk_ins.sum('restaurant')
}).to_dataframe().corr()

# Plot
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt='.2f'
)
plt.title('How Walk-ins Relate to Item Sales')


In [ ]:

conversion = (ds.sales.sum('item') / ds.walk_ins).rename('sales_per_walkin')

# Plot grid
conversion.hvplot.bar(
    x='date',
    by='restaurant',
    subplots=True,
    shared_axes=False,
    width=600,
    height=400,
    rot=45
).cols(2)

In [ ]:
ds.coords['day_name'] = ds.date.dt.strftime('%A')

# Pivot for heatmap
heatmap_data = ds.walk_ins.groupby('day_name').mean().to_dataframe().unstack()

plt.figure(figsize=(10, 6))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".0f",
    cmap="YlGnBu",
    linewidths=.5
)
plt.title('Average Walk-ins by Day of Week')
plt.ylabel('Restaurant')
plt.xlabel('')

In [ ]:
restaurant_avg = ds.walk_ins.mean(dim='date')

# Create figure with subplots
fig, axes = plt.subplots(nrows=len(ds.restaurant), figsize=(14, 10), sharex=True)

# Plot each restaurant separately
for i, restaurant in enumerate(ds.restaurant.values):
    # Get data for this restaurant
    walk_ins = ds.walk_ins.sel(restaurant=restaurant)
    avg = restaurant_avg.sel(restaurant=restaurant).item()

    # Create masks
    above_avg = walk_ins.where(walk_ins > avg)
    below_avg = walk_ins.where(walk_ins <= avg)

    # Plot
    axes[i].bar(
        below_avg.date.values,
        below_avg.values,
        color='lightgray',
        width=0.8
    )
    axes[i].bar(
        above_avg.date.values,
        above_avg.values,
        color='red',
        width=0.8
    )
    axes[i].axhline(
        y=avg,
        color='black',
        linestyle='--',
        linewidth=1
    )
    axes[i].set_title(f'{restaurant} (Avg: {avg:.0f} walk-ins)')
    axes[i].grid(True, axis='y', alpha=0.3)

# Final touches
plt.suptitle('Daily Walk-ins vs Restaurant-Specific Averages', y=1.02)
plt.xlabel('Date')
fig.text(0.04, 0.5, 'Walk-ins', va='center', rotation='vertical')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()